# Supervised Behavior-Cloning Training Loop

This notebook trains `DrivingModel` to imitate recorded BeamNG controls.

Pipeline position:

```text
collect_data.py -> raw_data.csv -> process_data.py -> processed_data.csv -> this notebook -> best_model.pth
```

Input features come from `state_schema.FEATURE_COLUMNS` through `DrivingDataset`.
Targets are steering, throttle, and brake.  This is regression, not classification, so the metric is MSE loss rather than accuracy.

In [ ]:
# Imports
# torch / nn: neural-network training
# pandas: CSV loading
# tqdm: progress bar during minibatch training
# train_test_split: reproducible 80/10/10 split
import torch
import torch.nn as nn
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader
from utils import DrivingDataset
from model import DrivingModel
from sklearn.model_selection import train_test_split

## Hyperparameters

These values are intentionally small enough for a class-project demo.  Increase `num_epochs` or collect more data if training is stable and time allows.


In [ ]:
# Number of full passes through the training set.
num_epochs = 20

# Minibatch size.  Smaller batches are noisier but can work well for small datasets.
batch_size = 8

# Shuffle only the training loader.  Validation/test loaders should stay deterministic.
shuffle = True

# Adam learning rate.  Kept conservative to avoid unstable updates.
lr = 0.0001

# Stop if validation loss fails to improve for this many epochs.
early_stopping = 5


## Load processed data and create datasets

`processed_data.csv` should come from `process_data.py`.  The split is 80% train, 10% validation, and 10% test.


In [ ]:
# Load training data.
csv = "processed_data.csv"
df = pd.read_csv(csv)

# First split: 80% train, 20% temporary validation/test pool.
train, test_val = train_test_split(
    df,
    random_state=42,
    test_size=0.2,
)

# Second split: split the 20% pool into 10% test and 10% validation.
test, val = train_test_split(
    test_val,
    random_state=42,
    test_size=0.5,
)

# DrivingDataset converts each row into (X, y) tensors.
train_ds = DrivingDataset(train)
test_ds = DrivingDataset(test)
val_ds = DrivingDataset(val)

# Shuffle only training data.  Evaluation loaders are deterministic.
train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle)
test_ld = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
val_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

print(f"train={len(train_ds)} test={len(test_ds)} val={len(val_ds)}")

## Create model, loss, and optimizer

The output controls are continuous values, so this is a regression task.  We use mean squared error loss.


In [ ]:
# Prefer GPU if available.  MPS supports Apple Silicon Macs.
if torch.cuda.is_available():
    dev = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"

device = torch.device(dev)
print(f"Running on device: {device}")

# DrivingModel expects 6 inputs and emits 3 controls.
model = DrivingModel().to(device)

# MSELoss compares predicted controls against recorded controls.
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

## Train with validation and early stopping

Correct PyTorch order per minibatch:

```text
zero_grad -> forward -> loss -> backward -> optimizer.step
```

Validation uses `model.eval()` and `torch.no_grad()` so dropout is disabled and gradients are not tracked.


In [ ]:
train_loss_list = []
val_loss_list = []

best_val_loss = float("inf")
patience = 0

for epoch in range(num_epochs):
    # Training phase: update model weights.
    model.train()
    running_loss = 0.0

    for X, y in tqdm(train_ld):
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_train_loss = running_loss / max(len(train_ld), 1)
    train_loss_list.append(epoch_train_loss)

    # Validation phase: measure generalization without updating weights.
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for X, y in val_ld:
            X = X.to(device)
            y = y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)
            val_running_loss += loss.item()

    epoch_val_loss = val_running_loss / max(len(val_ld), 1)
    val_loss_list.append(epoch_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f}"
    )

    # Save the best model based on validation loss.
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  -> New best val loss: {best_val_loss:.4f}, model saved.")
    else:
        patience += 1
        if patience >= early_stopping:
            print("Early stopping triggered")
            break


## Final test evaluation

The test set is held out until the end.  This gives a less biased estimate than repeatedly checking the validation set while training.

In [ ]:
best_model = DrivingModel().to(device)
best_model.load_state_dict(torch.load("best_model.pth", map_location=device))
best_model.eval()

test_running_loss = 0.0
with torch.no_grad():
    for X, y in test_ld:
        X = X.to(device)
        y = y.to(device)
        outputs = best_model(X)
        loss = criterion(outputs, y)
        test_running_loss += loss.item()

test_loss = test_running_loss / max(len(test_ld), 1)
print(f"Best Val Loss: {best_val_loss:.4f}")
print(f"Test Loss: {test_loss:.4f}")
